# CryptoRisk Lab — Adaptive GBM and GARCH(1,1) extension

**Independent, fixed-protocol retrospective study · 2026 January–October.** This notebook presents and independently audits the committed research extension, not a rewrite of the original 97-cell assignment notebook. The full, runnable model fit and Monte Carlo experiment are implemented in \`research/phase3_2026/run_research.py\`.

**Question:** Does updating volatility using a 30-day rolling estimate or a Gaussian GARCH(1,1) process improve 30-day BTC/USDT terminal-loss forecasts versus the original GBM and two-state Markov models?

**Primary metric:** Brier score for a terminal loss ≥10% exactly 30 days after forecast origin (lower is better). Thresholds ≥20%, ≥35% are secondary.

**Protocol first:** The method, models, thresholds and January–October 2026 evaluation period were documented in \`PRE_REGISTERED_PROTOCOL.md\` before retrieving any 2026 outcomes for this experiment. This is an independently separated **retrospective** evaluation, not a true prospective test. The original assignment notebook is unchanged.

## 1. What we are comparing

All models retain the historical 2022–2024 calibration where applicable. Forecasts begin on the selected daily 2026 origin; only that day's and earlier market information can update the current volatility state.

- **Constant GBM:** \(\log(S_{t+1}/S_t)=\mu+\sigma z_{t+1}\), using fixed 2022–2024 estimates for both mean and standard deviation. For a 30-day terminal threshold \(q\), the analytic loss probability is \(\Phi((\log(1-q)-30\mu)/(\sigma\sqrt{30}))\).
- **Two-state Markov:** Fixed, 2022–2024 calibrated regime-specific means, variances and transition probabilities; an origin's most recent 14 known returns determine the initial regime. The original 200,000-path probability for each starting regime is reused without changing its calibration.
- **EWMA:** \(v_{t+1}=0.9v_t+0.1(r_t-\mu)^2\). Current variance uses real returns known through \(t\); future variance uses **simulated future shocks only**.
- **Adaptive GBM:** Keep the original mean \(\mu\), but use the last 30 observed log returns to estimate the current daily standard deviation, held fixed during each projected 30-day forecast.
- **GARCH(1,1):** \(\varepsilon_t=r_t-\mu,\quad h_{t+1}=\omega+\alpha\varepsilon_t^2+\beta h_t\). Fit once on 2022–2024 demeaned log returns (percentage scale); update states only using past returns; simulate future shocks with the fitted Gaussian conditional variances.
- **Historical benchmark:** Fixed historical loss frequency, estimated on the calibration period, not tuned to 2025 or 2026.

All models predict the *same events*, using the *same forecast origin dates*. An adaptive model can be more responsive but less accurate.

In [ ]:
from pathlib import Path
import json, subprocess, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

REPO_URL = "https://github.com/it23372726/cryptorisk-lab.git"
RESEARCH_BRANCH = "research-adaptive-gbm-garch-2026-holdout"
ROOT = Path(".").resolve()
if not (ROOT / "research/phase3_2026/results/metrics_2026.json").exists():
    destination = Path("cryptorisk-lab")
    if not (destination / "research/phase3_2026/results/metrics_2026.json").exists():
        subprocess.run(["git","clone","--depth","1","--branch",RESEARCH_BRANCH,REPO_URL,str(destination)], check=True)
    ROOT = destination.resolve()
folder = ROOT / "research/phase3_2026/results"
metrics = json.loads((folder/"metrics_2026.json").read_text())
manifest = json.loads((folder/"archive_sha256_manifest.json").read_text())
ledger = pd.read_csv(folder/"forecast_ledger_2026.csv")
print("Source:", ROOT)
print("Frozen protocol:", metrics["protocol"])
print("Forecast origins:", metrics["origins"])
print("Evaluation:", metrics["evaluation_start"],"to",metrics["evaluation_last_origin"])
print("Final observed price:", metrics["last_actual_close"])
print("Archive SHA256 entries:", len(manifest["archives"]))
assert len(ledger)==274

## 2. Historical data and evidence protection

Data: Binance spot BTC/USDT UTC daily candles. The model training period is January 2022 through December 2024. Existing audited 2025 candles link training to 2026 without updating **fitted** parameters. The new holdout is January 1–October 31, 2026, with 274 daily forecast origins ending October 1.

The archival source URL, SHA-256 and month length are recorded for **58 monthly sources** (36 calibration months, 12 previously verified 2025 months, 10 separate 2026 months). GARCH coefficients are fitted **one time**, using the 2022–2024 sample. Future market prices are read only to determine the actual outcome 30 days later.

Because rolling 30-day forecasts overlap, 274 windows **cannot be counted as 274 independent trials**.

In [ ]:
garch = metrics["garch_fit"]
display(pd.DataFrame([{
    "omega (squared percentage return units)":garch["omega_pct2"],
    "alpha":garch["alpha"], "beta":garch["beta"],
    "alpha + beta":garch["persistence"],
    "Calibration returns":garch["n_train"],
    "Estimator converged":garch["fit_converged"]
}]))
assert garch["n_train"]==1095
assert garch["alpha"]+garch["beta"]<1
assert all(len(item["sha256"])==64 for item in manifest["archives"])
print("PASS: Training-only, stable Gaussian GARCH(1,1) and authenticated historical source manifest")

## 3. Recompute actual model scores

For each origin \(i\), the model estimates probability \(p_i\), and the actual loss outcome \(y_i\in\{0,1\}\) is **YES or NO**. The Brier score is

\[\mathrm{Brier}=\frac{1}{N}\sum_{i=1}^{N}(p_i-y_i)^2.\]

**Lower is better**, but a smaller sample score does not automatically imply a statistically significant or practically useful improvement. The original 2025 results have not been rewritten.

In [ ]:
names = ["historical","gbm","markov","ewma","adaptive_gbm","garch"]
scored = []
for threshold in (10,20,35):
    outcome = ledger[f"event_{threshold}"].to_numpy()
    actual = (1-ledger["btc_target"]/ledger["btc_origin"]>=threshold/100).astype(int).to_numpy()
    np.testing.assert_array_equal(outcome,actual)
    for name in names:
        forecast=ledger[f"{name}_p{threshold}"].to_numpy()
        computed=float(np.mean((forecast-outcome)**2))
        recorded=metrics["scores"][str(threshold)]["models"][name]["brier"]
        assert np.isclose(computed,recorded,atol=1e-10,rtol=0)
        scored.append({"Model":name,"Threshold":threshold,"Brier":computed,
                       "Mean forecast":float(np.mean(forecast)),
                       "Actual event count":int(outcome.sum())})
score_table=pd.DataFrame(scored)
display(score_table.pivot(index="Model",columns="Threshold",values="Brier").sort_values(10).round(6))
print("PASS: Every displayed Brier score reconstructed from individual daily predictions and real outcomes")

In [ ]:
primary=score_table[score_table.Threshold.eq(10)].sort_values("Brier")
fig,ax=plt.subplots(figsize=(10,4))
ax.barh(primary["Model"],primary["Brier"])
ax.invert_yaxis()
ax.set(title="Frozen 2026 holdout: 30-day loss ≥10% Brier score",xlabel="Brier score · lower = better")
ax.grid(axis="x",alpha=.25)
plt.tight_layout()
plt.show()

## 4. Do the dynamic probabilities anticipate 30-day losses?

The following curves are forecasts made separately at each 2026 origin. They are not actual prices and are not a trading signal. A volatile forecast curve is **not necessarily a useful curve**. We should evaluate its scores and calibration rather than favoring models that visibly move more.

In [ ]:
dates=pd.to_datetime(ledger.origin,utc=True)
fig,ax=plt.subplots(figsize=(13,4.5))
for name in ["gbm","markov","ewma","adaptive_gbm","garch"]:
    ax.plot(dates,ledger[f"{name}_p10"],linewidth=1.25,label=name)
for date in dates[ledger.event_10.eq(1)]:
    ax.axvline(date,alpha=.035)
ax.set(title="Forecast probability: 30-day BTC loss ≥10% · 2026 retrospective",
       xlabel="Forecast origin (UTC)",ylabel="Predicted probability")
ax.legend(ncol=3,fontsize=9)
ax.grid(alpha=.25)
plt.tight_layout()
plt.show()

## 5. Is an apparent accuracy difference statistically convincing?

**Primary candidate comparisons were fixed before examining 2026 outcomes**: adaptive GBM versus constant GBM, and Gaussian GARCH versus constant GBM. Both compare day-aligned Brier losses.

A **paired circular moving-block bootstrap**, using 30-day blocks, 3,000 fixed-seed resamples and 95% percentile intervals, accounts approximately for dependence between overlapping forecasts. Two primary tests are adjusted using Holm. The result is a *retrospective inferential check*, not proof of future outperformance. A positive candidate-minus-GBM Brier difference means the candidate scored **worse**.

In [ ]:
audit=[]
for k in ["adaptive_gbm_minus_gbm","garch_minus_gbm"]:
    difference=metrics["scores"]["10"]["differences"][k]
    primary=metrics["primary_holm_tests"][k]
    audit.append({"Comparison":k,
                  "Brier difference":difference["mean"],
                  "95% CI lower":difference["ci95"][0],
                  "95% CI upper":difference["ci95"][1],
                  "Holm-adjusted p":primary["holm_adjusted_p"],
                  "Supported improvement?":primary["claim_supported"]})
display(pd.DataFrame(audit).round(6))

## 6. Actual conclusion

In the fixed 2026 retrospective sample, **constant GBM recorded the lowest mean Brier score for a ≥10% terminal loss** (approximately 0.175239). GARCH was close (approximately 0.176664), Markov was approximately 0.178221, and both EWMA and 30-day adaptive GBM scored worse. Neither adaptive GBM nor GARCH demonstrated a statistically established improvement over constant GBM under the prespecified test.

**Interpretation:** Volatility adaptation can improve the *responsiveness* of probability forecasts, but responsiveness alone does not guarantee good loss-event prediction. GARCH's model assumptions, the chosen 30-day horizon, heavy-tailed BTC returns, overlapping event windows and the limited 2026 sample all restrict conclusions.

This is a meaningful negative scientific result, not a failure of the software. We should preserve it. It does not justify tuning more windows or models on 2026 and calling those new scores out-of-sample.

## 7. Full reproduction and sources

The scientific implementation and its tests are committed in the \`research/phase3_2026/\` directory. The following optional cell re-runs the **entire experiment** (new official Binance monthly downloads and the fitted GARCH; no changes to model choices). It is disabled for the default Colab *Run all* path so reading these already-verified results doesn't require repeated external downloads.

Source references:
- Original research notebook and main comparison: \`notebook/cryptorisk-lab.ipynb\`.
- Preregistered research plan: \`research/phase3_2026/PRE_REGISTERED_PROTOCOL.md\`.
- Binance public data: https://github.com/binance/binance-public-data
- Bollerslev (1986), GARCH(1,1), DOI: https://doi.org/10.1016/0304-4076(86)90063-1
- Engle (1982), conditional heteroskedasticity, DOI: https://doi.org/10.2307/1912773
- \`arch\` forecasting details: https://arch.readthedocs.io/en/latest/univariate/forecasting.html

AI-assisted code development and research drafting were used; all numerical claims must remain grounded in executed scripts, original notebooks, documented provenance and verified outputs.

In [ ]:
RECOMPUTE_FROM_OFFICIAL_ARCHIVES = False
if RECOMPUTE_FROM_OFFICIAL_ARCHIVES:
    subprocess.run([sys.executable,"-m","pip","install","-r",
                    str(ROOT/"research/phase3_2026/requirements-lock.txt")],check=True)
    subprocess.run([sys.executable,str(ROOT/"research/phase3_2026/run_research.py"),"--self-test"],check=True)
    subprocess.run([sys.executable,str(ROOT/"research/phase3_2026/run_research.py"),"--run"],check=True)
    subprocess.run([sys.executable,str(ROOT/"research/phase3_2026/check_results.py")],check=True)
    print("PASS: Full frozen 2026 experiment reproduced using original Binance archives")
else:
    print("Default: displayed committed, independently audited research results. Enable rerun explicitly above.")